In [1]:
import pandas as pd
import csv
import os

In [2]:
def strip_object_columns(df: pd.DataFrame) -> pd.DataFrame:

    for column in df.select_dtypes(include=['object']).columns:
        df[column] = df[column].astype(str).str.strip()
    
    return df

In [5]:
def clean_numeric_columns(df: pd.DataFrame, threshold: float = 0.7) -> pd.DataFrame: # adds a threshold of 0.7 (70%); so unless the row has 70% >= non-null values that are numeric, it will execute, if not, no conversion will be done on that row 
    df = df.copy()
    
    for col in df.columns:
        if col.lower() in {}:
            continue
        
        cleaned = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("$", "", regex=False)
            .str.replace("%", "", regex=False)
        )

        numeric = pd.to_numeric(cleaned, errors="coerce") # updated "ignore" to "coerce" due to FutureWarning

        non_null = cleaned.notna().sum()
        numeric_count = numeric.notna().sum()

        if non_null > 0 and numeric_count / non_null >= threshold: 
            df[col] = numeric
    
    return df

In [6]:
def remove_empty_rows(df: pd.DataFrame) -> pd.DataFrame: # this will remove the rows where the numeric columns are NaN
    numeric_cols = df.select_dtypes(include=['number']).columns
    
    if len(numeric_cols) == 0:
        return df # this checks the number of numeric rows; if 0, then it will end
    
    return df.dropna(subset=numeric_cols, how='all')

In [7]:
def format_numbers(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    def format_whole_numbers(x):
        if pd.isna(x):
            return x
        if isinstance(x, float) and x.is_integer():
            return int(x)
        return x

    numeric_cols = df.select_dtypes(include=['number']).columns
    df[numeric_cols] = df[numeric_cols].applymap(format_whole_numbers)

    return df

# function that formats numbers with no decimal value to whole integers and leaves float values untouched

In [8]:
def add_suffix_cleaned(file_name, suffix='_cleaned'): # this will add the suffix 'cleaned' to the file name separated by an underscore
    base, ext = file_name.rsplit(".", 1) 
    return f"{base}{suffix}.{ext}" # splits the name of the file into 'base' and 'ext'; adds '_cleaned' to the new file name before the file extension

In [11]:
def rename_csv_files(input_dir: str, output_dir: str, csv_rename_files: dict): # rename csv files with selective names; no blanket names

    for old_file, new_file in csv_rename_files.items():
        input_path = f"{input_dir}/{old_file}"
        output_path = f"{output_dir}/{new_file}"

        df = pd.read_csv(input_path)
        df.to_csv(output_path, index=False)

        print(f"Renamed {old_file} to {new_file}")

In [ ]:
def batch_nan_value_fix(df: pd.DataFrame) -> pd.DataFrame: # fixes the "NR" and "NA" values in the data files
    df.copy()
    df.replace({"NR": pd.NA, "NA": pd.NA}, inplace=True)

    return df

In [ ]:
def batch_clean_csv(input_ # cleans a batch of csv files and adds '_cleaned' before the .ext to show it is finalized
    for old_file, new_file in csv_files.items():
        input_file = f"{input_dir}/{old_file}"
        output_file = f"{output_dir}/{new_file}"

        df = pd.read_csv(input_file)   

        df = batch_nan_value_fix(df)
        df = strip_object_columns(df)
        df = clean_numeric_columns(df)
        df = remove_empty_rows(df)
        df = format_numbers(df)

        df.to_csv(output_file, index=False)
        print(f"Saved cleaned CSV: {output_file}")

In [ ]:
csv_files = {
    "2024 Affordability Rankings for Child Care Professionals with Children in Center-Based Child Care.csv": "center_ccp_child_rank_2024.csv",
    "2024 Average Annual Price of Full-Time Center-Based Child Care and Public College Tuition and Fees by State.csv": "center_pcollege_tuition_cost_2024.csv",
    "2024 Average Annual Price of Full-Time Center-Based Child Care by State.csv": "center_cost_state_2024.csv",
    "2024 Average Annual Price of Full-Time Family Child Care (FCC) by State.csv": "fcc_cost_state_2024.csv",
    "2024 Average Prices for Center-Based Child Care for Infants and Two Children Compared to Varying Poverty Levels.csv": "center_infants_twochild_cost_poverty_2024.csv",
    "2024 Average Prices for Center-Based Child Care for Toddlers and 4-Year-Olds Compared to Varying Poverty Levels.csv": "center_toddlers_4yearold_cost_poverty_2024.csv",
    "2024 Average Prices for Family Child Care (FCC) for Infants and Two Children Compared to Varying Poverty Levels.csv": "fcc_infants_twochild_cost_poverty_2024.csv",
    "2024 Average Prices for Family Child Care (FCC) for Toddlers and a 4-Year-Old Compared to Varying Poverty Levels.csv": "fcc_toddlers_4yearold_cost_poverty_2024.csv",
    "2024 Average Prices for Two Children in Center-Based Child Care Versus Median Housing Costs by State.csv": "center_twochild_cost_housing_cost_2024.csv",
    "2024 Ranking of Affordability of Center-Based Child Care for Single-Parent Households, School-Age and Two Children.csv": "center_singleparent_schoolage_twochild_rank_2024.csv",
    "2024 Ranking of Affordability of Center-Based Child Care for Single-Parent Households.csv": "center_singleparent_younger_than_schoolage_rank_2024.csv",
    "2024 Ranking of Least Affordable Center-Based Child Care for 4-Year-Olds.csv": "center_4yearold_rank_2024.csv",
    "2024 Ranking of Least Affordable Center-Based Child Care for Infants.csv": "center_infants_rank_2024.csv",
    "2024 Ranking of Least Affordable Center-Based Child Care for School-Age Children.csv": "center_schoolage_rank_2024.csv",
    "2024 Ranking of Least Affordable Center-Based Child Care for Toddlers.csv": "center_toddlers_rank_2024.csv",
    "2024 Ranking of Least Affordable Family Child Care (FCC) for 4-Year-Olds.csv": "fcc_4yearold_rank_2024.csv",
    "2024 Ranking of Least Affordable Family Child Care (FCC) for Infants.csv": "fcc_infants_rank_2024.csv",
    "2024 Ranking of Least Affordable Family Child Care (FCC) for School-Age Children.csv": "fcc_schoolage_rank_2024.csv",
    "2024 Ranking of Least Affordable Family Child Care (FCC) for Toddlers.csv": "fcc_toddlers_rank_2024.csv"
}

input_dir = "../data/child_care/working"
output_dir = "../data/child_care/working/test"

rename_csv_files(input_dir, output_dir, csv_files)

Renamed 2024 Affordability Rankings for Child Care Professionals with Children in Center-Based Child Care.csv to center_ccp_child_rank_2024.csv
Renamed 2024 Average Annual Price of Full-Time Center-Based Child Care and Public College Tuition and Fees by State.csv to center_pcollege_tuition_cost_2024.csv
Renamed 2024 Average Annual Price of Full-Time Center-Based Child Care by State.csv to center_cost_state_2024.csv
Renamed 2024 Average Annual Price of Full-Time Family Child Care (FCC) by State.csv to fcc_cost_state_2024.csv
Renamed 2024 Average Prices for Center-Based Child Care for Infants and Two Children Compared to Varying Poverty Levels.csv to center_infants_twochild_cost_poverty_2024.csv
Renamed 2024 Average Prices for Center-Based Child Care for Toddlers and 4-Year-Olds Compared to Varying Poverty Levels.csv to center_toddlers_4yearold_cost_poverty_2024.csv
Renamed 2024 Average Prices for Family Child Care (FCC) for Infants and Two Children Compared to Varying Poverty Levels.csv